# MCal Experiment: How Calibration Improves Explanation Quality

This notebook demonstrates that MCal calibration improves the quality of feature attribution explanations by:
1. Reducing missingness bias (KL divergence)
2. Improving sufficiency scores
3. Improving comprehensiveness scores

## 1. Setup and Imports

In [ ]:
import sys
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import timm  # For loading finetuned ViT models

# Add parent directory to path
sys.path.append('..')

# Import dataset classes and components
from experiments.all_data_loaders import MRIPatchedProbDataset, MRICleanDataset
from experiments.vision.mri_mcal_explanation_quality import load_mri_model
from src.calibrators.mcal_ce import SimpleMCalCE
from experiments.explanations import ImageLIME
from experiments.metrics import (
    ImageSufficiency, ImageComprehensiveness, ImageKLDivergence,
    compute_metric_improvement
)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

## 2. Configuration

In [ ]:
config = {
    'n_train': 2000,
    'n_test': 200,
    'ablation_rate': 0.5,
    'patch_size': 32,  # Changed to 32 to match finetuned models
    'explanation_samples': 100,
    'k_values': list(range(1, 26)),  # 1 to 25 for reasonable evaluation with 7x7 patch grid
    'batch_size': 32,
    'device': device
}

## 3. Load Model and Prepare Data

In [ ]:
# Load the model trained WITHOUT ablation (to demonstrate MCal's improvement)
import os
from pathlib import Path

# Use model trained with p_ablate=0.00 to show missingness bias
notebook_dir = Path.cwd()  
model_path = notebook_dir.parent / 'saved_models' / 'mri' / 'vit_mri_ps32_ablate0.00_best.pth'  # Model with missingness bias

if not model_path.exists():
    model_path = Path('../saved_models/mri/vit_mri_ps32_ablate0.00_best.pth')
    if not model_path.exists():
        raise FileNotFoundError(f"Model not found. Please ensure the model exists at: {model_path}")

print(f"Loading model from: {model_path}")

# Load checkpoint
checkpoint = torch.load(model_path, map_location=device)
print(f"Model trained for {checkpoint['epoch']+1} epochs with best val loss: {checkpoint['best_val_loss']:.4f}")
print(f"Model was trained with p_ablate={checkpoint['config']['p_ablate']:.2f} (clean data only - has missingness bias)")

# Create model and load state dict
base_model = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=4)
base_model.load_state_dict(checkpoint['model_state_dict'])
base_model = base_model.to(device)
base_model.eval()
uncal_model = base_model

# Create datasets with patch_size=32 (matching the finetuned model)
train_ablated_dataset = MRIPatchedProbDataset(
    split='train',
    n_samples=config['n_train'],
    p_ablate=config['ablation_rate'],
    patch_size=config['patch_size'],
    seed=42
)
test_dataset = MRICleanDataset(split='test', n_samples=config['n_test'])

# Create dataloaders
train_ablated_loader = DataLoader(train_ablated_dataset, batch_size=config['batch_size'], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False)

## 4. Train MCal Calibrator

In [ ]:
# Get labels and ablated predictions for training
train_labels = []
ablated_logits = []

with torch.no_grad():
    # Get predictions and labels from ablated data
    for images, labels in tqdm(train_ablated_loader, desc="Computing ablated predictions"):
        images = images.to(device)
        labels = labels.to(device)
        train_labels.append(labels)
        ablated_logits.append(base_model(images))
    
    train_labels = torch.cat(train_labels)
    ablated_logits = torch.cat(ablated_logits)

# Train MCal with default parameters
calibrator = SimpleMCalCE(num_classes=4).to(device)
stats = calibrator.fit(
    ablated_logits=ablated_logits,
    target_labels=train_labels,
    verbose=True
    # Using default max_steps and lr from SimpleMCalCE
)

print(f"Training complete! Final Loss: {stats['loss'][-1]:.4f}, Final Accuracy: {stats['acc'][-1]:.3f}")

In [ ]:
# Diagnostic: Check if the base model is already handling missingness well
print("Analyzing base model's response to missingness...")

# Test on a small batch to see prediction distributions
with torch.no_grad():
    # Get a batch of clean test data
    test_batch_images, test_batch_labels = next(iter(test_loader))
    test_batch_images = test_batch_images[:10].to(device)  # Just 10 samples
    test_batch_labels = test_batch_labels[:10].to(device)

    # Get predictions on clean data
    clean_logits = base_model(test_batch_images)
    clean_probs = torch.softmax(clean_logits, dim=1)

    # Create ablated version (same ablation as training)
    from experiments.all_data_loaders import _mask_random_patches_prob
    ablated_batch = []
    for img in test_batch_images:
        # Manually apply ablation to the image
        ablated_img = _mask_random_patches_prob(
            img.cpu(),
            mask_prob=config['ablation_rate'],
            patch_size=config['patch_size'],
            fill_val=0,
            seed=None  # Random ablation
        )
        ablated_batch.append(ablated_img)

    ablated_batch = torch.stack(ablated_batch).to(device)

    # Get predictions on ablated data
    ablated_logits = base_model(ablated_batch)
    ablated_probs = torch.softmax(ablated_logits, dim=1)

    # Calculate KL divergence between clean and ablated predictions
    kl_div = torch.nn.functional.kl_div(
        torch.log(ablated_probs + 1e-10),
        clean_probs,
        reduction='batchmean'
    )

    print(f"Base model KL divergence (clean vs ablated): {kl_div.item():.4f}")
    print(f"Clean accuracy: {(clean_probs.argmax(1) == test_batch_labels).float().mean():.2%}")
    print(f"Ablated accuracy: {(ablated_probs.argmax(1) == test_batch_labels).float().mean():.2%}")

    # Check prediction stability
    print(f"\nPrediction changes due to ablation:")
    pred_changes = (clean_probs.argmax(1) != ablated_probs.argmax(1)).float().mean()
    print(f"Percentage of predictions that changed: {pred_changes:.1%}")

    # Show how much the probabilities shift
    prob_shift = torch.abs(clean_probs - ablated_probs).mean()
    print(f"Average probability shift: {prob_shift:.4f}")

print("\nNote: If KL divergence is already very low (<0.01), the model is already robust to missingness.")
print("Additional calibration might not help or could even hurt.")

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(stats['loss'])
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss')
ax1.set_title('MCal Training Loss')
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

ax2.plot(stats['acc'])
ax2.set_xlabel('Step')
ax2.set_ylabel('Accuracy')
ax2.set_title('MCal Training Accuracy')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Generate All Explanations (LIME and SHAP)

In [ ]:
# Create calibrated model
calib_model = nn.Sequential(base_model, calibrator)
calib_model.eval()

# Load test data
test_images = []
test_labels = []
for images, labels in test_loader:
    test_images.append(images)
    test_labels.append(labels)
test_images = torch.cat(test_images).to(device)
test_labels = torch.cat(test_labels).to(device)

print(f"Loaded {len(test_images)} test samples")

In [ ]:
# Initialize ALL explainers (LIME and SHAP)
from experiments.explanations import ImageLIME, ImageKernelSHAP

print("Initializing explainers...")

# Use the same number of samples for both LIME and SHAP
explanation_samples = 100  # Same for both methods

# LIME explainers
uncal_lime_explainer = ImageLIME(
    model=uncal_model,
    num_samples=explanation_samples,
    patch_size=config['patch_size'],
    image_size=224
)

calib_lime_explainer = ImageLIME(
    model=calib_model,
    num_samples=explanation_samples,
    patch_size=config['patch_size'],
    image_size=224
)

# SHAP explainers (now using same number of samples as LIME)
uncal_shap_explainer = ImageKernelSHAP(
    model=uncal_model,
    num_samples=explanation_samples,
    patch_size=config['patch_size'],
    image_size=224
)

calib_shap_explainer = ImageKernelSHAP(
    model=calib_model,
    num_samples=explanation_samples,
    patch_size=config['patch_size'],
    image_size=224
)

print(f"✓ LIME explainers initialized with {explanation_samples} samples")
print(f"✓ SHAP explainers initialized with {explanation_samples} samples")

In [ ]:
# Load test data and generate explanations
test_images = []
test_labels = []
for images, labels in test_loader:
    test_images.append(images)
    test_labels.append(labels)
test_images = torch.cat(test_images).to(device)
test_labels = torch.cat(test_labels).to(device)

# Generate LIME explanations
uncal_attrs = []
calib_attrs = []

for i, (image, label) in enumerate(tqdm(zip(test_images, test_labels), total=len(test_images), desc="Generating explanations")):
    uncal_attrs.append(uncal_explainer.explain_instance(image, label.item()))
    calib_attrs.append(calib_explainer.explain_instance(image, label.item()))

## 6. Evaluate Metrics Across K Values

In [ ]:
# Initialize metrics
kl_metric = ImageKLDivergence(n_classes=4)
suff_metric = ImageSufficiency(patch_size=config['patch_size'], image_size=224)
comp_metric = ImageComprehensiveness(patch_size=config['patch_size'], image_size=224)

# Initialize results storage
results = {
    'k_values': config['k_values'],
    'missingness_bias': {'uncal': [], 'calib': []},
    'sufficiency': {'uncal': [], 'calib': []},
    'comprehensiveness': {'uncal': [], 'calib': []}
}

In [ ]:
# Evaluate LIME metrics across K values using pre-generated explanations
print("Evaluating LIME explanations across K values...")

for k in tqdm(config['k_values'], desc="Evaluating LIME k values"):
    suff_uncal_list = []
    suff_calib_list = []
    comp_uncal_list = []
    comp_calib_list = []
    masked_images_uncal = []
    masked_images_calib = []

    for i, (image, label) in enumerate(zip(test_images, test_labels)):
        label_idx = label.item()
        uncal_importance = uncal_lime_attrs[i]  # Use pre-generated LIME explanations
        calib_importance = calib_lime_attrs[i]  # Use pre-generated LIME explanations

        # Compute metrics
        suff_uncal_list.append(suff_metric.compute(uncal_model, image, uncal_importance, k, label_idx))
        suff_calib_list.append(suff_metric.compute(calib_model, image, calib_importance, k, label_idx))
        comp_uncal_list.append(comp_metric.compute(uncal_model, image, uncal_importance, k, label_idx))
        comp_calib_list.append(comp_metric.compute(calib_model, image, calib_importance, k, label_idx))

        # Create masked images for KL computation
        uncal_top_k = torch.argsort(torch.abs(uncal_importance), descending=True)[:k]
        calib_top_k = torch.argsort(torch.abs(calib_importance), descending=True)[:k]

        uncal_mask = suff_metric.create_mask_from_indices(uncal_top_k).to(device)
        calib_mask = suff_metric.create_mask_from_indices(calib_top_k).to(device)

        masked_img_uncal = image.clone()
        masked_img_calib = image.clone()
        for c in range(3):
            masked_img_uncal[c] = masked_img_uncal[c] * uncal_mask
            masked_img_calib[c] = masked_img_calib[c] * calib_mask

        masked_images_uncal.append(masked_img_uncal)
        masked_images_calib.append(masked_img_calib)

    # Compute KL divergence
    masked_images_uncal = torch.stack(masked_images_uncal)
    masked_images_calib = torch.stack(masked_images_calib)
    
    kl_uncal = kl_metric.compute(uncal_model, test_images, masked_images_uncal)
    kl_calib = kl_metric.compute(calib_model, test_images, masked_images_calib)

    # Store results
    results['missingness_bias']['uncal'].append(kl_uncal)
    results['missingness_bias']['calib'].append(kl_calib)
    results['sufficiency']['uncal'].append(np.mean(suff_uncal_list))
    results['sufficiency']['calib'].append(np.mean(suff_calib_list))
    results['comprehensiveness']['uncal'].append(np.mean(comp_uncal_list))
    results['comprehensiveness']['calib'].append(np.mean(comp_calib_list))

print("LIME evaluation complete!")

## 7. Visualize Results

In [ ]:
# Create plots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

k_vals = config['k_values']

# Plot 1: Missingness Bias
axes[0].plot(k_vals, results['missingness_bias']['uncal'], 'o-', label='Uncalibrated', linewidth=2)
axes[0].plot(k_vals, results['missingness_bias']['calib'], 's-', label='MCal Calibrated', color='green', linewidth=2)
axes[0].set_xlabel('Top-K Features Selected')
axes[0].set_ylabel('KL Divergence (log scale)')
axes[0].set_title('Missingness Bias')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

# Plot 2: Sufficiency
axes[1].plot(k_vals, results['sufficiency']['uncal'], 'o-', label='Uncalibrated', linewidth=2)
axes[1].plot(k_vals, results['sufficiency']['calib'], 's-', label='MCal Calibrated', color='green', linewidth=2)
axes[1].set_xlabel('Top-K Features Selected')
axes[1].set_ylabel('Sufficiency Score (↓ better)')
axes[1].set_title('Sufficiency')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Comprehensiveness
axes[2].plot(k_vals, results['comprehensiveness']['uncal'], 'o-', label='Uncalibrated', linewidth=2)
axes[2].plot(k_vals, results['comprehensiveness']['calib'], 's-', label='MCal Calibrated', color='green', linewidth=2)
axes[2].set_xlabel('Top-K Features Selected')
axes[2].set_ylabel('Comprehensiveness Score (↑ better)')
axes[2].set_title('Comprehensiveness')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('MCal Impact on LIME Explanation Quality', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Summary Statistics

In [ ]:
# Compute average improvements
avg_kl_improve = np.mean([
    compute_metric_improvement(u, c, 'kl_divergence')
    for u, c in zip(results['missingness_bias']['uncal'], results['missingness_bias']['calib'])
])

avg_suff_improve = np.mean([
    compute_metric_improvement(u, c, 'sufficiency')
    for u, c in zip(results['sufficiency']['uncal'], results['sufficiency']['calib'])
])

avg_comp_improve = np.mean([
    compute_metric_improvement(u, c, 'comprehensiveness')
    for u, c in zip(results['comprehensiveness']['uncal'], results['comprehensiveness']['calib'])
])

print("="*60)
print("SUMMARY: MCal Improvement on LIME Explanations")
print("="*60)
print(f"\nAverage Improvements Across All K Values:")
print(f"  Missingness Bias (KL): {avg_kl_improve:+.1f}%")
print(f"  Sufficiency: {avg_suff_improve:+.1f}%")
print(f"  Comprehensiveness: {avg_comp_improve:+.1f}%")

In [ ]:
# Evaluate SHAP metrics across K values using pre-generated explanations
print("Evaluating SHAP explanations across K values...")

# Initialize SHAP results storage
shap_results = {
    'k_values': config['k_values'],
    'missingness_bias': {'uncal': [], 'calib': []},
    'sufficiency': {'uncal': [], 'calib': []},
    'comprehensiveness': {'uncal': [], 'calib': []}
}

# Evaluate for each k value using SHAP explanations
for k in tqdm(config['k_values'], desc="Evaluating SHAP k values"):
    suff_uncal_list = []
    suff_calib_list = []
    comp_uncal_list = []
    comp_calib_list = []
    masked_images_uncal = []
    masked_images_calib = []

    # Use only the subset of images for which we have SHAP explanations
    for i, (image, label) in enumerate(zip(shap_test_images, shap_test_labels)):
        label_idx = label.item()
        uncal_importance = uncal_shap_attrs[i]
        calib_importance = calib_shap_attrs[i]

        # Compute metrics
        suff_uncal_list.append(suff_metric.compute(uncal_model, image, uncal_importance, k, label_idx))
        suff_calib_list.append(suff_metric.compute(calib_model, image, calib_importance, k, label_idx))
        comp_uncal_list.append(comp_metric.compute(uncal_model, image, uncal_importance, k, label_idx))
        comp_calib_list.append(comp_metric.compute(calib_model, image, calib_importance, k, label_idx))

        # Create masked images for KL computation
        uncal_top_k = torch.argsort(torch.abs(uncal_importance), descending=True)[:k]
        calib_top_k = torch.argsort(torch.abs(calib_importance), descending=True)[:k]

        uncal_mask = suff_metric.create_mask_from_indices(uncal_top_k).to(device)
        calib_mask = suff_metric.create_mask_from_indices(calib_top_k).to(device)

        masked_img_uncal = image.clone()
        masked_img_calib = image.clone()
        for c in range(3):
            masked_img_uncal[c] = masked_img_uncal[c] * uncal_mask
            masked_img_calib[c] = masked_img_calib[c] * calib_mask

        masked_images_uncal.append(masked_img_uncal)
        masked_images_calib.append(masked_img_calib)

    # Compute KL divergence
    masked_images_uncal = torch.stack(masked_images_uncal)
    masked_images_calib = torch.stack(masked_images_calib)
    
    kl_uncal = kl_metric.compute(uncal_model, shap_test_images, masked_images_uncal)
    kl_calib = kl_metric.compute(calib_model, shap_test_images, masked_images_calib)

    # Store results
    shap_results['missingness_bias']['uncal'].append(kl_uncal)
    shap_results['missingness_bias']['calib'].append(kl_calib)
    shap_results['sufficiency']['uncal'].append(np.mean(suff_uncal_list))
    shap_results['sufficiency']['calib'].append(np.mean(suff_calib_list))
    shap_results['comprehensiveness']['uncal'].append(np.mean(comp_uncal_list))
    shap_results['comprehensiveness']['calib'].append(np.mean(comp_calib_list))

print("SHAP evaluation complete!")

In [ ]:
# Compare LIME vs SHAP results
print("="*60)
print("COMPARISON: LIME vs SHAP Explanations")
print("="*60)

# Compute average improvements for SHAP
shap_avg_kl_improve = np.mean([
    compute_metric_improvement(u, c, 'kl_divergence')
    for u, c in zip(shap_results['missingness_bias']['uncal'], shap_results['missingness_bias']['calib'])
])

shap_avg_suff_improve = np.mean([
    compute_metric_improvement(u, c, 'sufficiency')
    for u, c in zip(shap_results['sufficiency']['uncal'], shap_results['sufficiency']['calib'])
])

shap_avg_comp_improve = np.mean([
    compute_metric_improvement(u, c, 'comprehensiveness')
    for u, c in zip(shap_results['comprehensiveness']['uncal'], shap_results['comprehensiveness']['calib'])
])

print(f"\nLIME-based MCal Improvements:")
print(f"  Missingness Bias (KL): {avg_kl_improve:+.1f}%")
print(f"  Sufficiency: {avg_suff_improve:+.1f}%")
print(f"  Comprehensiveness: {avg_comp_improve:+.1f}%")

print(f"\nSHAP-based MCal Improvements:")
print(f"  Missingness Bias (KL): {shap_avg_kl_improve:+.1f}%")
print(f"  Sufficiency: {shap_avg_suff_improve:+.1f}%")
print(f"  Comprehensiveness: {shap_avg_comp_improve:+.1f}%")

print("\n" + "="*60)
print("INSIGHT: MCal improvements should be consistent across")
print("different explanation methods, demonstrating that the")
print("calibration benefits are explanation-agnostic.")
print("="*60)

In [ ]:
# Visualize SHAP Results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

k_vals = config['k_values']

# Plot 1: Missingness Bias
axes[0].plot(k_vals, shap_results['missingness_bias']['uncal'], 'o-', label='Uncalibrated', linewidth=2, markersize=6)
axes[0].plot(k_vals, shap_results['missingness_bias']['calib'], 's-', label='MCal Calibrated', color='green', linewidth=2, markersize=6)
axes[0].set_xlabel('Top-K Features Selected', fontsize=12)
axes[0].set_ylabel('KL Divergence (log scale)', fontsize=12)
axes[0].set_title('Missingness Bias (SHAP)', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

# Plot 2: Sufficiency
axes[1].plot(k_vals, shap_results['sufficiency']['uncal'], 'o-', label='Uncalibrated', linewidth=2, markersize=6)
axes[1].plot(k_vals, shap_results['sufficiency']['calib'], 's-', label='MCal Calibrated', color='green', linewidth=2, markersize=6)
axes[1].set_xlabel('Top-K Features Selected', fontsize=12)
axes[1].set_ylabel('Sufficiency Score (↓ better)', fontsize=12)
axes[1].set_title('Sufficiency (SHAP)', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# Plot 3: Comprehensiveness
axes[2].plot(k_vals, shap_results['comprehensiveness']['uncal'], 'o-', label='Uncalibrated', linewidth=2, markersize=6)
axes[2].plot(k_vals, shap_results['comprehensiveness']['calib'], 's-', label='MCal Calibrated', color='green', linewidth=2, markersize=6)
axes[2].set_xlabel('Top-K Features Selected', fontsize=12)
axes[2].set_ylabel('Comprehensiveness Score (↑ better)', fontsize=12)
axes[2].set_title('Comprehensiveness (SHAP)', fontsize=14, fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.suptitle('MCal Impact on SHAP Explanation Quality', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Print numerical improvements for SHAP
shap_avg_kl_improve = np.mean([
    compute_metric_improvement(u, c, 'kl_divergence')
    for u, c in zip(shap_results['missingness_bias']['uncal'], shap_results['missingness_bias']['calib'])
])

shap_avg_suff_improve = np.mean([
    compute_metric_improvement(u, c, 'sufficiency')
    for u, c in zip(shap_results['sufficiency']['uncal'], shap_results['sufficiency']['calib'])
])

shap_avg_comp_improve = np.mean([
    compute_metric_improvement(u, c, 'comprehensiveness')
    for u, c in zip(shap_results['comprehensiveness']['uncal'], shap_results['comprehensiveness']['calib'])
])

print("\n" + "="*60)
print("SHAP-based MCal Improvements:")
print("="*60)
print(f"  Missingness Bias (KL): {shap_avg_kl_improve:+.1f}%")
print(f"  Sufficiency: {shap_avg_suff_improve:+.1f}%")
print(f"  Comprehensiveness: {shap_avg_comp_improve:+.1f}%")

## 10. Compare LIME vs SHAP Results

In [ ]:
# Create comprehensive comparison plot with both LIME and SHAP
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

k_vals = config['k_values']

# Row 1: LIME Results
# Plot 1: Missingness Bias (LIME)
axes[0,0].plot(k_vals, results['missingness_bias']['uncal'], 'o-', label='Uncalibrated', linewidth=2)
axes[0,0].plot(k_vals, results['missingness_bias']['calib'], 's-', label='MCal Calibrated', color='green', linewidth=2)
axes[0,0].set_xlabel('Top-K Features')
axes[0,0].set_ylabel('KL Divergence (log)')
axes[0,0].set_title('Missingness Bias (LIME)', fontweight='bold')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)
axes[0,0].set_yscale('log')

# Plot 2: Sufficiency (LIME)
axes[0,1].plot(k_vals, results['sufficiency']['uncal'], 'o-', label='Uncalibrated', linewidth=2)
axes[0,1].plot(k_vals, results['sufficiency']['calib'], 's-', label='MCal Calibrated', color='green', linewidth=2)
axes[0,1].set_xlabel('Top-K Features')
axes[0,1].set_ylabel('Sufficiency (↓ better)')
axes[0,1].set_title('Sufficiency (LIME)', fontweight='bold')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Plot 3: Comprehensiveness (LIME)
axes[0,2].plot(k_vals, results['comprehensiveness']['uncal'], 'o-', label='Uncalibrated', linewidth=2)
axes[0,2].plot(k_vals, results['comprehensiveness']['calib'], 's-', label='MCal Calibrated', color='green', linewidth=2)
axes[0,2].set_xlabel('Top-K Features')
axes[0,2].set_ylabel('Comprehensiveness (↑ better)')
axes[0,2].set_title('Comprehensiveness (LIME)', fontweight='bold')
axes[0,2].legend()
axes[0,2].grid(True, alpha=0.3)

# Row 2: SHAP Results
# Plot 4: Missingness Bias (SHAP)
axes[1,0].plot(k_vals, shap_results['missingness_bias']['uncal'], 'o-', label='Uncalibrated', linewidth=2)
axes[1,0].plot(k_vals, shap_results['missingness_bias']['calib'], 's-', label='MCal Calibrated', color='green', linewidth=2)
axes[1,0].set_xlabel('Top-K Features')
axes[1,0].set_ylabel('KL Divergence (log)')
axes[1,0].set_title('Missingness Bias (SHAP)', fontweight='bold')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)
axes[1,0].set_yscale('log')

# Plot 5: Sufficiency (SHAP)
axes[1,1].plot(k_vals, shap_results['sufficiency']['uncal'], 'o-', label='Uncalibrated', linewidth=2)
axes[1,1].plot(k_vals, shap_results['sufficiency']['calib'], 's-', label='MCal Calibrated', color='green', linewidth=2)
axes[1,1].set_xlabel('Top-K Features')
axes[1,1].set_ylabel('Sufficiency (↓ better)')
axes[1,1].set_title('Sufficiency (SHAP)', fontweight='bold')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

# Plot 6: Comprehensiveness (SHAP)
axes[1,2].plot(k_vals, shap_results['comprehensiveness']['uncal'], 'o-', label='Uncalibrated', linewidth=2)
axes[1,2].plot(k_vals, shap_results['comprehensiveness']['calib'], 's-', label='MCal Calibrated', color='green', linewidth=2)
axes[1,2].set_xlabel('Top-K Features')
axes[1,2].set_ylabel('Comprehensiveness (↑ better)')
axes[1,2].set_title('Comprehensiveness (SHAP)', fontweight='bold')
axes[1,2].legend()
axes[1,2].grid(True, alpha=0.3)

plt.suptitle('MCal Impact on Explanation Quality: LIME vs SHAP Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Compare LIME vs SHAP results
print("="*60)
print("COMPARISON: LIME vs SHAP Explanations")
print("="*60)

# Compute average improvements for SHAP
shap_avg_kl_improve = np.mean([
    compute_metric_improvement(u, c, 'kl_divergence')
    for u, c in zip(shap_results['missingness_bias']['uncal'], shap_results['missingness_bias']['calib'])
])

shap_avg_suff_improve = np.mean([
    compute_metric_improvement(u, c, 'sufficiency')
    for u, c in zip(shap_results['sufficiency']['uncal'], shap_results['sufficiency']['calib'])
])

shap_avg_comp_improve = np.mean([
    compute_metric_improvement(u, c, 'comprehensiveness')
    for u, c in zip(shap_results['comprehensiveness']['uncal'], shap_results['comprehensiveness']['calib'])
])

print(f"\nLIME-based MCal Improvements:")
print(f"  Missingness Bias (KL): {avg_kl_improve:+.1f}%")
print(f"  Sufficiency: {avg_suff_improve:+.1f}%")
print(f"  Comprehensiveness: {avg_comp_improve:+.1f}%")

print(f"\nSHAP-based MCal Improvements:")
print(f"  Missingness Bias (KL): {shap_avg_kl_improve:+.1f}%")
print(f"  Sufficiency: {shap_avg_suff_improve:+.1f}%")
print(f"  Comprehensiveness: {shap_avg_comp_improve:+.1f}%")

print("\n" + "="*60)
print("KEY INSIGHT:")
print("Both LIME and SHAP show similar improvements from MCal,")
print("validating that MCal's benefits are explanation-method agnostic.")
print("="*60)

In [ ]:
# Evaluate SHAP metrics
shap_results = {
    'k_values': config['k_values'],
    'missingness_bias': {'uncal': [], 'calib': []},
    'sufficiency': {'uncal': [], 'calib': []},
    'comprehensiveness': {'uncal': [], 'calib': []}
}

print("Evaluating SHAP-based metrics...")
for k in tqdm(config['k_values'], desc="Evaluating SHAP k values"):
    suff_uncal_list = []
    suff_calib_list = []
    comp_uncal_list = []
    comp_calib_list = []
    masked_images_uncal = []
    masked_images_calib = []

    for i, (image, label) in enumerate(zip(shap_test_images, shap_test_labels)):
        label_idx = label.item()
        uncal_importance = uncal_shap_attrs[i]
        calib_importance = calib_shap_attrs[i]

        # Compute metrics
        suff_uncal_list.append(suff_metric.compute(uncal_model, image, uncal_importance, k, label_idx))
        suff_calib_list.append(suff_metric.compute(calib_model, image, calib_importance, k, label_idx))
        comp_uncal_list.append(comp_metric.compute(uncal_model, image, uncal_importance, k, label_idx))
        comp_calib_list.append(comp_metric.compute(calib_model, image, calib_importance, k, label_idx))

        # Create masked images for KL computation
        uncal_top_k = torch.argsort(torch.abs(uncal_importance), descending=True)[:k]
        calib_top_k = torch.argsort(torch.abs(calib_importance), descending=True)[:k]

        uncal_mask = suff_metric.create_mask_from_indices(uncal_top_k).to(device)
        calib_mask = suff_metric.create_mask_from_indices(calib_top_k).to(device)

        masked_img_uncal = image.clone()
        masked_img_calib = image.clone()
        for c in range(3):
            masked_img_uncal[c] = masked_img_uncal[c] * uncal_mask
            masked_img_calib[c] = masked_img_calib[c] * calib_mask

        masked_images_uncal.append(masked_img_uncal)
        masked_images_calib.append(masked_img_calib)

    # Compute KL divergence
    masked_images_uncal = torch.stack(masked_images_uncal)
    masked_images_calib = torch.stack(masked_images_calib)
    
    kl_uncal = kl_metric.compute(uncal_model, shap_test_images, masked_images_uncal)
    kl_calib = kl_metric.compute(calib_model, shap_test_images, masked_images_calib)

    # Store results
    shap_results['missingness_bias']['uncal'].append(kl_uncal)
    shap_results['missingness_bias']['calib'].append(kl_calib)
    shap_results['sufficiency']['uncal'].append(np.mean(suff_uncal_list))
    shap_results['sufficiency']['calib'].append(np.mean(suff_calib_list))
    shap_results['comprehensiveness']['uncal'].append(np.mean(comp_uncal_list))
    shap_results['comprehensiveness']['calib'].append(np.mean(comp_calib_list))

In [ ]:
# Import and initialize SHAP explainers
from experiments.explanations import ImageSHAP

# Use fewer samples for SHAP due to computational cost
shap_samples = 50

uncal_shap_explainer = ImageSHAP(
    model=uncal_model,
    num_samples=shap_samples,
    patch_size=config['patch_size'],
    image_size=224
)

calib_shap_explainer = ImageSHAP(
    model=calib_model,
    num_samples=shap_samples,
    patch_size=config['patch_size'],
    image_size=224
)

print(f"Initialized SHAP explainers with {shap_samples} samples")
print(f"Patch size: {config['patch_size']}x{config['patch_size']}")